#  1 : Load and Inspect Predictions  

In [ ]:
# Step 1 — Import libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# Load predictions
df = pd.read_csv("../data/roberta_large_weighted_predictions.csv")
print(" Data loaded successfully!")
print(df.head(20))
print(df.shape)


#### Observation

From these examples, we can see that the model performs well on clear emotional statements such as *happiness*, *sadness*, and *surprise*, but it struggles with more ambiguous or context-dependent sentences.  
Misclassifications mostly occur in **neutral** sentences being predicted as emotional ones — for example, a narration line about a contestant's dilemma (neutral → fear) or a short dismissive remark (neutral → anger).  
This suggests that the model interprets certain neutral words like *dilemma* or dismissive phrases like *don’t feel bad* as emotional cues.  

Similarly, sentences such as a short remark predicting another player's downfall (fear → anger) and “Go!” (anger → happiness) show that the model can be influenced by **strong wording or punctuation**, often mistaking emotional intensity for a different emotion category.  
Overall, the model tends to **overpredict high-intensity emotions** (like anger or happiness) and underrepresent subtle or context-driven emotions (like fear or neutral), highlighting the challenge of capturing nuanced emotional meaning in text without tone or context.

-----------------------

# 2 : Check Model Accuracy and Correctness

 
Because our dataset is imbalanced (some emotions appear far more often than others), **F1-score** gives a fairer picture than accuracy.  
Here we calculate accuracy, macro F1, and weighted F1 to understand overall model performance.


In [ ]:
from sklearn.metrics import f1_score, accuracy_score

df["is_correct"] = df["true_label"] == df["pred_label"]

# Overall metrics
acc = accuracy_score(df["true_label"], df["pred_label"])
macro_f1 = f1_score(df["true_label"], df["pred_label"], average="macro")
weighted_f1 = f1_score(df["true_label"], df["pred_label"], average="weighted")

print(f" Accuracy: {acc:.3f}")
print(f" Macro F1 (equal per class): {macro_f1:.3f}")
print(f" Weighted F1 (weighted by frequency): {weighted_f1:.3f}")


### Results Interpretation  

####   The Weighted RoBERTa-Large model achieved strong overall accuracy (69.3%) 
####   and a macro F1 of 0.632, showing balanced performance across classes.
####   Since the dataset is imbalanced, macro F1 is our primary evaluation metric,
####   as it treats each emotion equally regardless of frequency.
####   The model performs best on frequent classes like "neutral" and "happiness,"
####   but struggles with rare classes such as "disgust" and "surprise."
####   This pattern explains why the macro F1 is lower than the weighted F1 and accuracy.


--------------------


# 3 : Confusion Matrix (Emotion Overlaps)

In [ ]:
labels = sorted(df["true_label"].unique())

cm = confusion_matrix(df["true_label"], df["pred_label"], labels=labels)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.title("Confusion Matrix — RoBERTa-Large Emotion Classification")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()


### Confusion Matrix — RoBERTa-Large (Weighted)

The confusion matrix shows how accurately the **Weighted RoBERTa-Large** model predicted each emotion.  
Dark diagonal cells represent correct predictions; off-diagonal ones indicate confusions between emotions.

####  Observations
- **Happiness** and **Neutral** are predicted most accurately.  
- **Neutral → Happiness:** Slightly positive neutral sentences are read as happy.  
- **Fear → Neutral:** Fearful sentences without strong emotion words are seen as neutral.  
- **Sadness → Neutral/Happiness:** Some sad tones are missed or softened.  
- **Surprise → Happiness/Neutral:** Overlaps in expression cause confusion.  
- **Disgust:** Rare and poorly learned due to very few examples.

####  Summary
The model handles frequent emotions well but struggles with rare or subtle ones, especially *disgust* and *surprise*.  
This explains why the **macro F1 (0.632)** is lower than the **accuracy (0.693)** and **weighted F1 (0.692)**.

---------------

# 4 : Per-Class Metrics (Precision, Recall, F1)

#### Here we check how well the model performs for each emotion.We look at **precision**, **recall**, and **F1-score** for every class. This helps us see which emotions are easy or hard for the model to predict.

In [ ]:
from sklearn.metrics import classification_report

# Generate detailed metrics for each emotion
report = classification_report(df["true_label"], df["pred_label"], digits=3)
print(report)


### – Per-Class Metrics Summary

The table shows precision, recall, and F1-score for each emotion class.  
Performance is strongest for **happiness (F1 = 0.77)** and **neutral (F1 = 0.72)**, where the model consistently identifies clear emotional tone.  
It performs moderately on **fear** and **sadness**, while **anger** and **surprise** remain harder to classify due to overlapping cues and limited examples.  
**Disgust** appears weak and unreliable because of the very small number of samples (only 6).

Overall, the model handles frequent emotions well but struggles with rare or ambiguous ones.  
This imbalance explains why the **macro F1 (0.632)** is lower than the **accuracy (0.693)** and **weighted F1 (0.692)**.

------------------------------------

# 5 : Most Common Confusions

### Here we check which emotion pairs are confused most often.  

In [ ]:
# Find wrong predictions
errors = df[df["is_correct"] == False]

# Count confusion pairs
error_counts = (
    errors.groupby(["true_label", "pred_label"])
          .size()
          .reset_index(name="count")
          .sort_values("count", ascending=False)
)
error_counts.head(10)


### Common Confusions (Results)

The table above shows the emotion pairs most often confused by the model.  
The most frequent mix-ups occur between **neutral** and **happiness**, where slightly positive neutral sentences are classified as happy.  
Other common confusions include **fear → neutral** and **sadness → fear or happiness**.

### What This Means
- Many neutral sentences sound mildly positive (e.g., “It’s fine,” “That’s okay”), leading to **neutral → happiness** errors.  
- **Fear** and **sadness** often share neutral wording or lack strong emotional cues.  
- **Anger** and **surprise** sometimes overlap with happiness due to similar emotional intensity.  

These patterns highlight why the **macro F1 (0.632)** is lower than the **accuracy (0.693)** — the model struggles most with subtle or overlapping emotional tones.


------------------------------

# 6 : Sentence Length vs Accuracy

#### We check if sentence length affects prediction quality. Short sentences often lack context, so the model may guess wrong.  Longer sentences give more clues, leading to better accuracy.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Make sure sentence_length exists
if "sentence_length" not in df.columns:
    df["sentence_length"] = df["sentence"].apply(lambda x: len(str(x).split()))

# Calculate mean sentence length for correct vs incorrect predictions
mean_lengths = (
    df.groupby("is_correct", as_index=False)["sentence_length"]
      .mean()
      .rename(columns={"is_correct": "Prediction", "sentence_length": "Avg_Length"})
)

# Map True/False to clearer labels
mean_lengths["Prediction"] = mean_lengths["Prediction"].map({True: "Correct", False: "Incorrect"})

# Plot bar chart
plt.figure(figsize=(5,4))
sns.barplot(x="Prediction", y="Avg_Length", data=mean_lengths, palette="Blues_d")

# Add numbers above bars
for index, row in mean_lengths.iterrows():
    plt.text(index, row.Avg_Length + 0.2, f"{row.Avg_Length:.1f}", ha="center", fontsize=10)

plt.title("Average Sentence Length by Prediction Outcome — RoBERTa-Large")
plt.xlabel("Prediction Outcome")
plt.ylabel("Average Sentence Length (words)")
plt.ylim(0, mean_lengths["Avg_Length"].max() + 2)
plt.tight_layout()
plt.show()


#### Observation

The bar chart shows that incorrectly predicted sentences are, on average, **longer (9.8 words)** than correctly predicted ones (**8.4 words**).  
This suggests that longer or more complex sentences make emotion classification harder for the model.  
Such sentences often contain multiple emotional cues, neutral phrases, or mixed tones, which can confuse the classifier.  
In contrast, shorter sentences with clear emotional expressions are easier for the model to interpret correctly.

---------------------------

# 7 : Common Words in Wrong Predictions

#### Now we check which words appear most often in sentences the model got wrong. These words might be neutral or unclear, making emotions harder to detect.

In [ ]:
from collections import Counter

# Combine all wrong sentences into one string
error_words = " ".join(errors["sentence"]).lower().split()

# Count most common words
common_error_words = Counter(error_words).most_common(20)
common_error_words


###  Observation

Most words in wrong predictions are very common and neutral, like “to”, “I”, “you”, and “the”.  
These words don’t show emotion, so the model can’t easily guess the correct label.  
This shows that the model needs more emotional or context-rich clues to make better predictions.

------------------------------------------------

# 8 : Errors per True Emotion

In [ ]:
errors["true_label"].value_counts().plot(kind="bar", title="Errors per True Emotion")


#### Observation

The bar chart shows the number of misclassified samples for each true emotion.  
The model makes the most errors on **neutral**, **sadness**, and **fear** classes.  
This pattern suggests that these emotions are more ambiguous or overlap frequently with others — for example, neutral sentences with slight positivity are often misread as happiness.  
In contrast, **disgust** and **surprise** show fewer errors, but this is mainly because they have very few examples in the dataset, not because the model performs better on them.  
Overall, the model struggles most with emotions that have subtle or context-dependent expressions.

------------------------------------